In [ ]:
# CSV 파일 불러오기

import pandas as pd
from pathlib import Path

csv_path = Path(r"C:\Users\2022A00520\Desktop\raw_ko.csv")
df = pd.read_csv(csv_path, encoding="utf-8-sig")

print("전체 문서 수:", len(df))
print("열 이름:", df.columns.tolist())
print("그룹별 문서 수:")
print(df["group"].value_counts())
df[["group", "title", "text"]].head(3)

In [ ]:
# 1. 정제
# 중복 데이터 제거 + 노이즈 제거

import re
import html
import unicodedata

def clean_text(text):
    text = unicodedata.normalize("NFKC", html.unescape(text)) # HTML 엔티티, 유니코드 정규화
    text = text.translate(str.maketrans({"‘": "'", "’": "'", "“": '"', "”": '"'})) 
    text = re.sub(r"(?is)<(script|style)\b[^>]*>.*?</\1>", " ", text) # HTML script/style 태그
    text = re.sub(r"<[^>]+>", " ", text)                    # HTML 태그
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)      # URL
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", " ", text)  # 이메일
    text = re.sub(r"^\s*\[[^\]\n]*(?:기자|뉴스엔)[^\]\n]*\]", " ", text) # 기자 이름이 포함된 작성자 표시 줄
    text = re.sub(r"(?m)^[ \t]*뉴스엔[^\r\n]*\[비식별\][^\r\n]*$", " ", text) # 비식별 표시 줄
    text = re.sub(r"[\ufeff\u200b-\u200f]", " ", text)       # 보이지 않는 문자
    text = re.sub(r"\s+", " ", text).strip()
    return text

before = len(df)
df.drop_duplicates(subset=["text"], inplace=True)
print("본문이 완전히 같은 중복 문서 제거:", before - len(df))

df["clean_text"] = df["text"].apply(clean_text)

before = len(df)
df = df[df["clean_text"] != ""]
df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

print("정제 후 빈 문서/추가 중복 제거:", before - len(df))
print("정제 후 문서 수:", len(df))
df[["title", "clean_text"]].head(3)

In [ ]:
# 2. 정규화
# 그룹/멤버/팬덤/소속사/앨범/곡명과 전문용어 통일
# 같은 대상을 뜻하는 여러 표기를 대표 표기로 바꿉니다.

normalization_dict = {
    # 분석 대상 그룹명
    "방탄소년단": ["bts", "방탄 소년단", "방탄", "bangtan sonyeondan", "bangtan boys"],
    "블랙핑크": ["blackpink", "black pink", "블랙 핑크", "블핑"],
    "세븐틴": ["seventeen", "svt"],
    "트와이스": ["twice"],
    "베이비몬스터": ["babymonster", "baby monster", "baby-monster", "베이비 몬스터", "베몬"],
    "코르티스": ["cortis"],

    # 방탄소년단 멤버: 영문명/활동명/본명
    "알엠": ["rm", "rap monster", "랩몬스터", "랩 몬스터", "김남준"],
    "진": ["jin", "김석진"],
    "슈가": ["suga", "agust d", "어거스트 디", "어거스트디", "민윤기"],
    "제이홉": ["j-hope", "j hope", "jhope", "제이 홉", "정호석"],
    "지민": ["jimin", "박지민"],
    "뷔": ["v", "김태형"],
    "정국": ["jungkook", "jung kook", "전정국"],

    # 블랙핑크 멤버
    "지수": ["jisoo", "김지수"],
    "제니": ["jennie", "김제니"],
    "로제": ["rosé", "rose", "박채영"],
    "리사": ["lisa", "라리사", "lalisa"],

    # 세븐틴 멤버
    "에스쿱스": ["s.coups", "s coups", "scoups", "최승철"],
    "정한": ["jeonghan", "윤정한"],
    "조슈아": ["joshua", "홍지수"],
    "준": ["jun", "문준휘"],
    "호시": ["hoshi", "권순영"],
    "원우": ["wonwoo", "전원우"],
    "우지": ["woozi", "이지훈"],
    "디에잇": ["the8", "the 8", "디 에잇", "서명호"],
    "민규": ["mingyu", "김민규"],
    "도겸": ["dk", "dokyeom", "이석민"],
    "승관": ["seungkwan", "부승관"],
    "버논": ["vernon", "최한솔"],
    "디노": ["dino", "이찬"],

    # 트와이스 멤버
    "나연": ["nayeon", "임나연"],
    "정연": ["jeongyeon", "유정연"],
    "모모": ["momo", "히라이 모모"],
    "사나": ["sana", "미나토자키 사나"],
    "지효": ["jihyo", "박지효"],
    "미나": ["mina", "묘이 미나"],
    "다현": ["dahyun", "김다현"],
    "채영": ["chaeyoung", "손채영"],
    "쯔위": ["tzuyu", "저우쯔위"],

    # 베이비몬스터 / 코르티스 멤버
    "루카": ["ruka"],
    "파리타": ["pharita"],
    "아사": ["asa"],
    "아현": ["ahyeon", "정아현"],
    "라미": ["rami", "하람", "신하람"],
    "로라": ["rora", "이다인"],
    "치키타": ["chiquita"],
    "마틴": ["martin"],
    "제임스": ["james"],
    "주훈": ["juhoon"],
    "성현": ["seonghyeon"],
    "건호": ["keonho"],

    # 팬덤명
    "아미": ["army", "a.r.m.y"],
    "블링크": ["blink"],
    "캐럿": ["carat"],
    "원스": ["once"],
    "몬스티즈": ["monstiez"],
    "코어": ["coer"],

    # 소속사명: 모회사와 자회사는 서로 다른 대표 표기로 유지
    "하이브": ["hybe"],
    "빅히트뮤직": ["big hit music", "bighit music", "bighit", "big hit entertainment",
                 "빅히트 뮤직", "빅히트엔터테인먼트", "빅히트 엔터테인먼트", "빅히트"],
    "와이지엔터테인먼트": ["yg entertainment", "yg엔터테인먼트", "yg 엔터테인먼트", "yg", "와이지 엔터테인먼트"],
    "제이와이피엔터테인먼트": ["jyp entertainment", "jyp엔터테인먼트", "jyp 엔터테인먼트", "jyp", "제이와이피 엔터테인먼트"],
    "플레디스엔터테인먼트": ["pledis entertainment", "pledis", "플레디스 엔터테인먼트", "플레디스"],
    "에스엠엔터테인먼트": ["sm entertainment", "sm엔터테인먼트", "sm 엔터테인먼트", "sm"],
    "오드아틀리에": ["odd atelier", "oddatelier"],

    # 주요 앨범명/곡명: 방탄소년단
    "다이너마이트": ["dynamite"],
    "버터": ["butter"],
    "퍼미션투댄스": ["permission to dance", "퍼미션 투 댄스"],
    "작은것들을위한시": ["boy with luv", "작은 것들을 위한 시"],
    "디엔에이": ["dna"],
    "페이크러브": ["fake love", "페이크 러브"],
    "마이크드롭": ["mic drop", "마이크 드롭"],
    "아이돌": ["idol"],
    "윙스": ["wings", "윙즈"],
    "러브유어셀프": ["love yourself", "러브 유어셀프", "러브 유어 셀프"],
    "맵오브더솔": ["map of the soul", "맵 오브 더 솔"],
    "유네버워크얼론": ["you never walk alone", "유 네버 워크 얼론", "유 네버 워크 어론"],
    "낫투데이": ["not today", "낫 투데이"],
    "아이니드유": ["i need u", "아이 니드 유"],
    "런": ["run"],
    "불타오르네": ["fire"],
    "아리랑": ["arirang"],
    "스윔": ["swim"],
    "세븐": ["seven"],
    "호프온더스테이지": ["hope on the stage", "호프 온 더 스테이지"],

    # 주요 앨범명/곡명: 블랙핑크
    "하우유라이크댓": ["how you like that", "하우 유 라이크 댓"],
    "킬디스러브": ["kill this love", "킬 디스 러브"],
    "뚜두뚜두": ["ddu-du ddu-du"],
    "러브식걸즈": ["lovesick girls", "러브식 걸즈"],
    "핑크베놈": ["pink venom", "핑크 베놈"],
    "셧다운": ["shut down", "셧 다운"],
    "본핑크": ["born pink", "본 핑크"],
    "더앨범": ["the album", "더 앨범"],
    "스퀘어원": ["square one", "스퀘어 원"],
    "스퀘어투": ["square two", "스퀘어 투"],
    "스퀘어업": ["square up", "스퀘어 업"],
    "아이스크림": ["ice cream"],
    "사워캔디": ["sour candy", "사워 캔디"],
    "뛰어": ["jump"],
    "데드라인": ["deadline"],
    "아파트": ["apt.", "apt"],

    # 주요 앨범명/곡명: 세븐틴
    "아주나이스": ["very nice", "아주 나이스"],
    "울고싶지않아": ["don't wanna cry", "울고 싶지 않아"],
    "레프트앤드라이트": ["left & right", "left and right", "레프트 앤드 라이트"],
    "달링": ["darl+ing", "darl'ing", "darling"],
    "손오공": ["super"],
    "음악의신": ["god of music", "음악의 신"],
    "마에스트로": ["maestro"],
    "올원": ["al1"],
    "다이아몬드엣지": ["diamond edge", "다이아몬드 엣지"],
    "비더선": ["be the sun", "비 더 선"],
    "유어초이스": ["your choice", "유어 초이스"],
    "아타카": ["attacca"],

    # 주요 앨범명/곡명: 트와이스
    "낙낙": ["knock knock", "낙 낙"],
    "치어업": ["cheer up", "치어 업"],
    "시그널": ["signal"],
    "라이키": ["likey"],
    "하트셰이커": ["heart shaker", "하트 셰이커"],
    "왓이즈러브": ["what is love", "왓 이즈 러브"],
    "댄스더나잇어웨이": ["dance the night away", "댄스 더 나잇 어웨이"],
    "예스오어예스": ["yes or yes", "예스 오어 예스"],
    "팬시": ["fancy"],
    "필스페셜": ["feel special", "필 스페셜"],
    "모어앤드모어": ["more & more", "more and more", "모어 앤드 모어"],
    "알코올프리": ["alcohol-free", "alcohol free", "알코올 프리"],
    "테이스트오브러브": ["taste of love", "테이스트 오브 러브"],
    "캔디팝": ["candy pop", "캔디 팝"],
    "더필즈": ["the feels", "더 필즈"],
    "레디투비": ["ready to be", "레디 투 비"],
    "디스이즈포": ["this is for", "디스 이즈 포"],
    "미사모": ["misamo"],

    # 주요 앨범명/곡명: 베이비몬스터 / 코르티스
    "배터업": ["batter up", "배터 업"],
    "쉬시": ["sheesh"],
    "드립": ["drip"],
    "스턱인더미들": ["stuck in the middle", "스턱 인 더 미들"],
    "포에버": ["forever"],
    "위고업": ["we go up", "위 고 업"],
    "사이코": ["psycho"],
    "춤": ["choom"],
    "리얼리라이크유": ["really like you", "리얼리 라이크 유"],
    "슈가허니아이스티": ["sugar honey ice tea", "슈가 허니 아이스 티"],
    "아이라이크잇": ["i like it", "아이 라이크 잇"],
    "베이비몬스터앨범": ["babymons7er"],  # 동명 앨범은 그룹명 불용어와 구분
    "컬러아웃사이드더라인스": ["color outside the lines", "컬러 아웃사이드 더 라인스"],
    "레드레드": ["redred", "레드 레드"],
    "그린그린": ["greengreen", "그린 그린"],
    "왓유원트": ["what you want", "왓 유 원트"],
    "패션": ["fashion"],

    # 전문용어/유의미한 외국어/띄어쓰기 통일
    "케이팝": ["k-pop", "kpop", "k pop", "k팝", "k-팝", "k 팝"],
    "뮤직비디오": ["mv", "m/v", "뮤비", "뮤직 비디오", "music video"],
    "소셜미디어": ["sns", "social media", "소셜 미디어"],
    "월드투어": ["world tour", "월드 투어"],
    "팬사인회": ["팬 사인회", "팬 싸인회", "팬싸인회"],
    "팬미팅": ["fan meeting", "fanmeeting", "팬 미팅"],
    "빌보드": ["billboard"],
    "유튜브": ["youtube", "you tube"],
    "스포티파이": ["spotify"],
    "틱톡": ["tiktok", "tik tok"],
    "위버스": ["weverse"],
    "브이라이브": ["v live", "vlive", "v라이브"],
    "케이콘": ["kcon"],
    "마마어워즈": ["mama awards", "mama", "마마 어워즈"],
    "더팩트뮤직어워즈": ["tma", "더팩트 뮤직 어워즈"],
    "음악방송": ["음악 방송", "음방"],
    "포브스": ["forbes"],
    "힙합": ["hip hop", "hip-hop"],
    "사이키델릭록": ["psychedelic rock", "사이키델릭 록"],

    # 기사에 함께 등장하는 다른 그룹: 분석 대상 6개 그룹과는 구분
    "엑소": ["exo"],
    "엔시티": ["nct"],
    "갓세븐": ["got7"],
    "스트레이키즈": ["stray kids", "스트레이 키즈"],
    "투모로우바이투게더": ["tomorrow x together", "txt"],
    "에스파": ["aespa"],
    "아이브": ["ive"],
    "르세라핌": ["le sserafim"],
    "캣츠아이": ["katseye"],
    "더보이즈": ["the boyz"],
}

# BE/ON/GO처럼 일반 영단어와 겹치는 짧은 제목은 따옴표/괄호 안에서만 변환합니다.
short_title_dict = {"be": "비", "on": "온", "go": "고", "r": "알", "tt": "티티"}

# 영문 대소문자를 통일하고, 긴 표기를 먼저 찾아 부분 치환을 방지합니다.
alias_dict = {}
for standard, aliases in normalization_dict.items():
    for alias in [standard] + aliases:
        alias_dict[alias.lower()] = standard

patterns = []
for alias in sorted(alias_dict, key=len, reverse=True):
    pattern = re.escape(alias)
    if re.search(r"[a-z0-9]", alias):
        pattern = r"(?<![a-z0-9])" + pattern + r"(?![a-z0-9])"
    patterns.append(pattern)
normalization_pattern = re.compile("|".join(patterns))

def normalize_text(text):
    text = text.lower()
    for english, korean in short_title_dict.items():
        pattern = r"(?<=['\"(\[])\s*" + re.escape(english) + r"\s*(?=['\")\]])"
        text = re.sub(pattern, korean, text)
    text = normalization_pattern.sub(lambda match: alias_dict[match.group()], text)
    # 방탄소년단(BTS), BTS(방탄소년단) -> 방탄소년단(방탄소년단) -> 방탄소년단
    text = re.sub(r"([가-힣]+)\s*\(\s*\1\s*\)", r"\1", text)
    # 정규화가 끝난 뒤 남은 기호를 제거합니다.
    text = re.sub(r"[^가-힣a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["normalized_text"] = df["clean_text"].apply(normalize_text)
df["group_normalized"] = df["group"].apply(normalize_text)

print("정규화 예시:",normalize_text("BTS(방탄소년단)의 MV, K-pop, SNS, 월드 투어, 팬 사인회"))

df[["clean_text", "normalized_text"]].head(3)


In [ ]:
# 3. 형태소 분석: Kiwi 사용

from kiwipiepy import Kiwi

kiwi = Kiwi()

# 정규화한 복합 고유명사와 뉴스 복합어를 사용자 사전에 등록합니다.
user_words = set(normalization_dict) | set(short_title_dict.values())
user_words |= {"뉴스엔", "기사제보", "보도자료", "비식별", "무단전재", "재배포", "포토엔", "포토엔에이치디"}

for word in sorted(user_words):
    if len(word) >= 2:
        kiwi.add_user_word(word, "NNP")

# 사용자 사전 등록은 단어 인식을 돕지만, 모든 문장에서 한 단어로 인식되는 것을 보장하지는 않습니다.
def pos_tagging(text):
    return [(token.form, token.tag) for token in kiwi.tokenize(text)]

df["pos"] = df["normalized_text"].apply(pos_tagging)

print("형태소:", [word for word, tag in df["pos"].iloc[0]][:40])
print()
print("품사 태깅 결과:", df["pos"].iloc[0][:40])


In [ ]:
# 4. 불용어 제거: Kiwi 기본 불용어 + 뉴스용 불용어 + 그룹명

from kiwipiepy.utils import Stopwords

stopwords = Stopwords()
basic_stopwords = stopwords.stopwords.copy()
basic_stoptags = stopwords.stoptags.copy()

news_stopwords = [
    "기자", "사진", "제공", "보도", "뉴스", "기사", "매체", "취재", "자료", "관계자",
    "뉴스엔", "보도자료", "기사제보", "제보", "무단전재", "무단", "전재", "재배포", "금지",
    "저작권", "copyright", "비식별", "포토엔", "포토엔에이치디", "씨", "측"
]

# CSV의 분석 대상인 6개 그룹명을 모든 문서에서 제거합니다.
group_stopwords = df["group_normalized"].unique().tolist()
news_stopword_pairs = {(word, tag) for word in news_stopwords for tag in ["NNG", "NNP"]}
group_stopword_pairs = {(word, tag) for word in group_stopwords for tag in ["NNG", "NNP"]}
stopwords.add(news_stopword_pairs | group_stopword_pairs)

# 이후 품사 선택을 위해 단어와 품사를 함께 남깁니다.
df["pos_no_stopwords"] = df["pos"].apply(lambda tokens: [(word, tag) for word, tag in tokens if (word, tag) not in stopwords])

# 적용한 불용어를 생략 없이 확인합니다.
stopword_rows = []
for word, tag in sorted(stopwords.stopwords):
    sources = []
    if (word, tag) in basic_stopwords:
        sources.append("Kiwi 기본")
    if (word, tag) in news_stopword_pairs:
        sources.append("뉴스용")
    if (word, tag) in group_stopword_pairs:
        sources.append("그룹명")
    stopword_rows.append([word, tag, ", ".join(sources)])

for tag in sorted(stopwords.stoptags):
    stopword_rows.append(["(해당 품사 전체)", tag, "Kiwi 기본"])

stopword_table = pd.DataFrame(stopword_rows, columns=["단어", "품사", "사전"])
print(stopword_table.to_string(index=False))
print("제거하는 분석 대상 그룹명:", sorted(group_stopwords))
print("불용어 제거 전:", df["pos"].iloc[0][:40])
print()
print("불용어 제거 후:", df["pos_no_stopwords"].iloc[0][:40])


In [ ]:
# 5. 품사 선택: 일반명사 + 고유명사 + 유의미한 외국어 변환

foreign_dict = {
    "bts": "방탄소년단", "blackpink": "블랙핑크", "seventeen": "세븐틴",
    "twice": "트와이스", "babymonster": "베이비몬스터", "cortis": "코르티스",
    "billboard": "빌보드", "youtube": "유튜브", "spotify": "스포티파이",
    "tiktok": "틱톡", "weverse": "위버스"
}

def select_pos(tokens):
    result = []
    for word, tag in tokens:
        if tag in ["NNG", "NNP"]:
            result.append((word, tag))
        elif tag == "SL" and word.lower() in foreign_dict:
            result.append((foreign_dict[word.lower()], "NNP"))
    return result

df["selected_pos"] = df["pos_no_stopwords"].apply(select_pos)
df["nouns"] = df["selected_pos"].apply(lambda tokens: [word for word, tag in tokens])

print("품사 선택 전:", df["pos_no_stopwords"].iloc[0][:40])
print()
print("품사 선택 후:", df["nouns"].iloc[0][:40])


In [ ]:
# 6. 등장한 문서 수가 적은 단어 제거 + CSV 저장

from collections import Counter
import json

# 서로 다른 문서 3개 이상에서 등장한 단어만 남깁니다.
MIN_DOC_FREQ = 3

document_frequency = Counter()
for tokens in df["nouns"]:
    document_frequency.update(set(tokens))

vocabulary = {word for word, count in document_frequency.items() if count >= MIN_DOC_FREQ}
df["tokens"] = df["nouns"].apply(lambda tokens: [word for word in tokens if word in vocabulary])

before = len(df)
df = df[df["tokens"].str.len() > 0].reset_index(drop=True)
df["processed_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))
df["token_count"] = df["tokens"].str.len()

# 이후 분석에 사용할 문서별 토큰 리스트입니다.
texts = df["tokens"].tolist()

# 중간 형태소 열은 제외하고 최종 토큰 리스트는 JSON 문자열로 저장합니다.
result = df.drop(columns=["pos", "pos_no_stopwords", "selected_pos", "nouns"]).copy()
result["tokens"] = result["tokens"].apply(lambda tokens: json.dumps(tokens, ensure_ascii=False))

output_path = Path(r"C:\Users\2022A00520\Desktop\processed_ko.csv")
result.to_csv(output_path, index=False, encoding="utf-8-sig")

print("문서 빈도 기준:", MIN_DOC_FREQ, "개 이상 문서")
print("서로 다른 단어 수:", len(document_frequency), "->", len(vocabulary))
print("토큰이 없어 제거한 문서 수:", before - len(df))
print("최종 문서 수:", len(df))
print("저장 위치:", output_path)
df[["group_normalized", "title", "tokens", "token_count"]].head(3)
